# 01 — Data Cleaning

**Input:** `data/raw/sfu_ml.db` 
**Output:** `data/processed/sfu_clean.db` — table: `offerings`

### SFU Section Code Structure

SFU section codes follow the pattern: **[letter][digits]** e.g. `D100`, `E200`, `C100`

The leading letter is the delivery mode:

| Code | Meaning | Primary? |
|------|---------|----------|
| D | Day | ✅ |
| E | Evening | ✅ |
| C | Distance Education | ✅ |
| F | French | ✅ |
| J | SFU NOW | ✅ |
| V | Video conference | ✅ |
| O | Online | ✅ |

Non-primary sections use **multi-letter prefixes**: `LAB`, `TUT`, `STD`, `OPL`, `RQL`, `STL`, `SEM`, `PRA`, `WKS`, `FLD`, `OLC`, `INS`

**Filter rule:** keep rows where `section_code` matches `^[A-Z]\d` — single letter followed by a digit.

### Cleaning Steps
| Step | Filter | Reason |
|------|--------|---------|
| 1 | `section_code` matches `^[A-Z]\d` | Remove labs, tutorials, seminars, practicum etc. |
| 2 | `ml_course_id IS NOT NULL` AND `ml_term_id IS NOT NULL` | Remove sections with no course or term metadata |
| 3 | `capacity > 0` | Remove placeholders and unconfigured sections |
| 4 | `enrolled IS NOT NULL AND enrolled > 0` | Remove sections with no published enrollment — overenrolled rows (enrolled > capacity) are kept |
| 5 | Normalize `instructor` | Replace TBA / blank / whitespace-only with NULL |
| 6 | Drop `waitlist` column | Near-zero variance — no signal for modeling |

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

RAW_DB   = Path('../data/raw/sfu_ml.db')
CLEAN_DB = Path('../data/processed/sfu_clean.db')
CLEAN_DB.parent.mkdir(exist_ok=True)

assert RAW_DB.exists(), f'not found: {RAW_DB}'
print('ready')

ready


---
## 1. Load raw data

In [2]:
conn = sqlite3.connect(RAW_DB)

raw = pd.read_sql("""
    SELECT
        o.offering_id,
        o.ml_course_id,
        o.ml_term_id,
        o.dept_code,
        o.course_number,
        o.section_code,
        o.instructor,
        o.campus,
        o.capacity,
        o.enrolled,
        o.waitlist,
        c.course_level,
        c.degree_level,
        c.units,
        c.prereq_count,
        c.title,
        t.year,
        t.term,
        t.term_order,
        t.semester_code,
        t.is_covid_affected
    FROM ml_section_offerings o
    LEFT JOIN ml_courses c ON o.ml_course_id = c.ml_course_id
    LEFT JOIN ml_terms   t ON o.ml_term_id   = t.ml_term_id
""", conn)

conn.close()
print(f'raw: {len(raw):,} rows  x  {raw.shape[1]} cols')

raw: 53,503 rows  x  21 cols


---
## 2. Understand what section codes actually exist

In [3]:
import re

raw['prefix'] = raw['section_code'].str.extract(r'^([A-Za-z]+)', expand=False).str.upper()
raw['prefix'].value_counts().head(30)

prefix
D      27478
G      17480
I       2298
E       1910
A       1288
OL      1079
C        631
B        613
LA       278
F        167
J         84
LB        63
LC        38
BLS       33
P         28
L         20
OP         7
GO         2
OPL        2
LAB        2
HUM        1
Z          1
Name: count, dtype: int64

In [4]:
raw['is_single_letter'] = raw['prefix'].str.len() == 1
raw.groupby('is_single_letter')['prefix'].value_counts().to_frame('count')

count
is_single_letter prefix       
False            OL       1079
                 LA        278
                 LB         63
                 LC         38
                 BLS        33
                 OP          7
                 GO          2
                 LAB         2
                 OPL         2
                 HUM         1
True             D       27478
                 G       17480
                 I        2298
                 E        1910
                 A        1288
                 C         631
                 B         613
                 F         167
                 J          84
                 P          28
                 L          20
                 Z           1

---
## 3. Apply cleaning steps

In [5]:
# Step 1 — primary sections only
step1 = raw[raw['section_code'].str.match(r'^[A-Z]\d', na=False)]
print(f'after step 1 (primary only):                    {len(step1):>7,}  removed {len(raw)-len(step1):,}')

after step 1 (primary only):                     51,998  removed 1,505


In [6]:
# confirm what delivery modes survived
step1['prefix'].value_counts()

prefix
D    27478
G    17480
I     2298
E     1910
A     1288
C      631
B      613
F      167
J       84
P       28
L       20
Z        1
Name: count, dtype: int64

In [7]:
# Step 2 — matched courses AND matched terms
# Both foreign keys must resolve — rows with null ml_term_id have no temporal context
# and cannot be used for any time-based feature engineering
step2 = step1[
    step1['ml_course_id'].notna() &
    step1['ml_term_id'].notna()
]
print(f'after step 2 (matched courses + terms):         {len(step2):>7,}  removed {len(step1)-len(step2):,}')

after step 2 (matched courses + terms):          47,417  removed 4,581


In [8]:
# Step 3 — valid capacity
step3 = step2[(step2['capacity'].notna()) & (step2['capacity'] > 0)]
print(f'after step 3 (capacity > 0):                    {len(step3):>7,}  removed {len(step2)-len(step3):,}')

after step 3 (capacity > 0):                     46,776  removed 641


In [9]:
# Step 4 — published enrollment
# Keep enrolled > capacity (overenrolled) — that's real data, not an error
# Drop only null or zero — those are placeholder / unconfigured sections
step4 = step3[(step3['enrolled'].notna()) & (step3['enrolled'] > 0)]
print(f'after step 4 (enrolled not null and > 0):       {len(step4):>7,}  removed {len(step3)-len(step4):,}')

# sanity check — how many overenrolled rows survived?
overenrolled = step4[step4['enrolled'] > step4['capacity']]
print(f'  overenrolled rows kept (enrolled > capacity): {len(overenrolled):>7,}')

after step 4 (enrolled not null and > 0):        35,285  removed 11,491
  overenrolled rows kept (enrolled > capacity):     999


In [10]:
# Step 5 — normalize instructor
# TBA, blank, whitespace-only → None (will be stored as NULL in SQLite)
# This is not a filter — no rows are dropped, just values replaced
step5 = step4.copy()
step5['instructor'] = step5['instructor'].str.strip()
step5['instructor'] = step5['instructor'].replace({'TBA': None, '': None})

tba_count = step4['instructor'].str.strip().isin(['TBA', '']).sum()
null_count = step5['instructor'].isna().sum()
print(f'instructor values normalized to NULL: {tba_count:,}')
print(f'total NULL instructors after step 5:  {null_count:,}  ({null_count/len(step5)*100:.1f}%)')

instructor values normalized to NULL: 0
total NULL instructors after step 5:  5,605  (15.9%)


In [11]:
# Step 6 — drop waitlist column and helpers
# waitlist data is nearly empty — no signal for modeling
step6 = step5.drop(columns=['waitlist', 'prefix', 'is_single_letter'])

# check for nulls in columns we need to cast to int
# units can be NULL in ml_courses for some entries — those rows are unusable
int_cols = ['ml_course_id', 'ml_term_id', 'course_level', 'units']
null_counts = step6[int_cols].isnull().sum()
print('Null counts before int cast:')
print(null_counts)
print()

# drop rows with null in any int column (units is the likely offender)
before = len(step6)
step6 = step6.dropna(subset=int_cols)
dropped = before - len(step6)
if dropped > 0:
    print(f'Dropped {dropped:,} rows with null in: {int_cols}')

# now safe to cast
step6 = step6.copy()
step6['ml_course_id'] = step6['ml_course_id'].astype(int)
step6['ml_term_id']   = step6['ml_term_id'].astype(int)
step6['course_level'] = step6['course_level'].astype(int)
step6['units']        = step6['units'].astype(int)

print(f'after step 6 (waitlist dropped, dtypes fixed):  {len(step6):>7,}  cols: {step6.shape[1]}')
print()
print(step6[['ml_course_id','ml_term_id','course_level','units']].dtypes)

Null counts before int cast:
ml_course_id       0
ml_term_id         0
course_level       0
units           1626
dtype: int64

Dropped 1,626 rows with null in: ['ml_course_id', 'ml_term_id', 'course_level', 'units']
after step 6 (waitlist dropped, dtypes fixed):   33,659  cols: 20

ml_course_id    int64
ml_term_id      int64
course_level    int64
units           int64
dtype: object


In [12]:
# full summary
clean = step6.copy()

print('=== CLEANING SUMMARY ===')
print(f'Raw rows:                         {len(raw):>8,}')
print(f'Removed — non-primary sections:   {len(raw)-len(step1):>8,}')
print(f'Removed — unmatched course/term:  {len(step1)-len(step2):>8,}')
print(f'Removed — zero capacity:          {len(step2)-len(step3):>8,}')
print(f'Removed — null/zero enrolled:     {len(step3)-len(step4):>8,}')
print(f'Removed — null units/course_level:{len(step5)-len(step6):>8,}')
print(f'Normalized — instructor to NULL:  {tba_count:>8,}  (rows kept)')
print(f'─────────────────────────────────────────')
print(f'Clean rows:                       {len(clean):>8,}  ({len(clean)/len(raw)*100:.1f}% retained)')
print(f'Columns:                          {clean.shape[1]:>8}')

=== CLEANING SUMMARY ===
Raw rows:                           53,503
Removed — non-primary sections:      1,505
Removed — unmatched course/term:     4,581
Removed — zero capacity:               641
Removed — null/zero enrolled:       11,491
Removed — null units/course_level:   1,626
Normalized — instructor to NULL:         0  (rows kept)
─────────────────────────────────────────
Clean rows:                         33,659  (62.9% retained)
Columns:                                20


---
## 4. Verify

In [13]:
# no nulls in critical columns
critical = ['ml_course_id', 'ml_term_id', 'dept_code', 'course_number',
            'section_code', 'capacity', 'enrolled', 'course_level', 'units']
clean[critical].isnull().sum()

ml_course_id     0
ml_term_id       0
dept_code        0
course_number    0
section_code     0
capacity         0
enrolled         0
course_level     0
units            0
dtype: int64

In [14]:
# only single-letter section codes remain
clean['section_code'].str[0].value_counts()

section_code
D    19527
G     9956
E     1690
A      811
C      560
B      547
I      335
F      147
J       58
P       14
L       13
Z        1
Name: count, dtype: int64

In [15]:
# rows per term — summer dip pattern should still be present
(
    clean.groupby(['year', 'term', 'term_order'])
    .size()
    .reset_index(name='n')
    .sort_values(['year', 'term_order'])
    .drop(columns='term_order')
)

,year,term,n
1,2020,spring,1934
2,2020,summer,1250
0,2020,fall,1886
4,2021,spring,1990
5,2021,summer,1251
3,2021,fall,1950
7,2022,spring,1985
8,2022,summer,1295
6,2022,fall,1905
10,2023,spring,2010


In [16]:
# spot check: CMPT 225 across all terms
(
    clean[(clean['dept_code'] == 'CMPT') & (clean['course_number'] == '225')]
    [['year', 'term', 'section_code', 'capacity', 'enrolled', 'instructor']]
    .sort_values(['year', 'term'])
)

,year,term,section_code,capacity,enrolled,instructor
6173,2020,fall,D100,200,157,"Mitchell, David"
6174,2020,fall,D200,200,143,"Shermer, Thomas"
868,2020,spring,D100,184,182,"Mitchell, David"
869,2020,spring,D200,144,115,"Edgar, John"
3711,2020,summer,D100,200,191,"Edgar, John"
3712,2020,summer,D200,150,87,"Taheri Gharagozloo, Pooya"
14712,2021,fall,D100,200,116,"Mitchell, David"
14713,2021,fall,D200,200,140,"Imran, Hazra; Shermer, Thomas"
9216,2021,spring,D100,200,156,"Shinkar, Igor"
9217,2021,spring,D200,160,115,"Thomas, Jack"


In [17]:
# check if any dept+term combinations lost ALL sections after cleaning
raw_depts_per_term   = raw.groupby(['dept_code', 'year', 'term']).size().reset_index(name='raw_n')
clean_depts_per_term = clean.groupby(['dept_code', 'year', 'term']).size().reset_index(name='clean_n')

merged = raw_depts_per_term.merge(clean_depts_per_term, on=['dept_code','year','term'], how='left')
lost   = merged[merged['clean_n'].isna()]

print(f'dept+term combinations that lost all sections: {len(lost)}')
if len(lost) > 0:
    print(lost['dept_code'].value_counts().head(20))

dept+term combinations that lost all sections: 128
dept_code
FEP      19
LBRL     19
GRAD     18
ARAB     12
UGRAD    12
NEUR     11
NUSC      6
GERM      5
APMA      5
SPAN      5
EDPR      4
GRK       3
PLCY      3
PERS      2
PLAN      2
SD        2
Name: count, dtype: int64


In [18]:
# if any departments lost all their sections, check what section codes they used
if len(lost) > 0:
    lost_depts = lost['dept_code'].unique()
    print('Section codes used by departments that lost all sections:')
    print(
        raw[raw['dept_code'].isin(lost_depts)]
        .groupby(['dept_code', 'prefix'])['offering_id']
        .count()
        .sort_values(ascending=False)
        .head(30)
    )

Section codes used by departments that lost all sections:
dept_code  prefix
EDPR       G         348
FEP        D         226
PLCY       G         186
LBRL       G         114
GRAD       G          96
NUSC       D          77
GERM       D          71
PLAN       D          60
LBRL       I          57
APMA       G          53
SD         D          41
SPAN       B          33
           D          26
GRK        B          23
NEUR       G          20
PLCY       I          15
ARAB       D          14
GRK        D          14
UGRAD      D          14
PLCY       D          12
SD         E          11
           OL         11
SPAN       L           9
PERS       D           7
PLAN       E           6
           B           5
GERM       L           4
PLAN       OL          2
PLCY       OL          2
SD         C           2
Name: offering_id, dtype: int64


---
## 5. Save to processed DB

In [19]:
if CLEAN_DB.exists():
    CLEAN_DB.unlink()

conn = sqlite3.connect(CLEAN_DB)

conn.execute("""
    CREATE TABLE offerings (
        offering_id       INTEGER PRIMARY KEY,
        ml_course_id      INTEGER NOT NULL,
        ml_term_id        INTEGER NOT NULL,
        dept_code         TEXT,
        course_number     TEXT,
        section_code      TEXT,
        instructor        TEXT,
        campus            TEXT,
        capacity          INTEGER,
        enrolled          INTEGER,
        course_level      INTEGER,
        degree_level      TEXT,
        units             INTEGER,
        prereq_count      INTEGER,
        title             TEXT,
        year              INTEGER,
        term              TEXT,
        term_order        INTEGER,
        semester_code     INTEGER,
        is_covid_affected INTEGER
    )
""")

clean.to_sql('offerings', conn, index=False, if_exists='append')
conn.commit()
conn.close()

print(f'saved → {CLEAN_DB}')
print(f'rows:   {len(clean):,}')
print(f'cols:   {clean.shape[1]}')

saved → ..\data\processed\sfu_clean.db
rows:   33,659
cols:   20


In [20]:
# verify the saved DB reads back correctly
conn  = sqlite3.connect(CLEAN_DB)
check = pd.read_sql('SELECT COUNT(*) as n FROM offerings', conn)
cols  = pd.read_sql('PRAGMA table_info(offerings)', conn)
conn.close()

print(f'rows in sfu_clean.db: {check["n"].iloc[0]:,}')
print()
print('columns saved:')
print(cols[['name', 'type']].to_string(index=False))

rows in sfu_clean.db: 33,659

columns saved:
             name    type
      offering_id INTEGER
     ml_course_id INTEGER
       ml_term_id INTEGER
        dept_code    TEXT
    course_number    TEXT
     section_code    TEXT
       instructor    TEXT
           campus    TEXT
         capacity INTEGER
         enrolled INTEGER
     course_level INTEGER
     degree_level    TEXT
            units INTEGER
     prereq_count INTEGER
            title    TEXT
             year INTEGER
             term    TEXT
       term_order INTEGER
    semester_code INTEGER
is_covid_affected INTEGER
